# BehaviorDataset Test
Verify that the BehaviorDataset class loads and queries data correctly.

In [4]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../")

import numpy as np
import matplotlib.pyplot as plt
from tools.behavior import BehaviorDataset
from tools.params import Params

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load Test Session

In [2]:
# Load the test session: M103_2026_02_18_15_30
session_name = "M103_2026_02_18_15_30"
print(f"Loading session: {session_name}...")
dataset = BehaviorDataset(session_name)
print(dataset)

Loading session: M103_2026_02_18_15_30...
M103_2026_02_18_15_30
fields: ['values_before_camera_trigger', 'idx_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
Repairing columns ['MotSen1_X', 'MotSen1_Y']
Extending index to 41999 in trial: free and id: 996, inserting NaN.
Extending index to 41999 in trial: free and id: 996, inserting NaN.
Combined every 1 bins
Resulting all_spikes ephys data shape is (NxT): (15, 48000)
Resulting CP_spikes ephys data shape is (NxT): (267, 48000

## 2. Inspect Session Properties

In [9]:
print(f"Animal: {dataset.animal_id}")
print(f"Condition: {dataset.condition}")
print(f"Available directions: {dataset.directions}")
print(f"\nTrials per direction:")
for direction in dataset.directions:
    print(f"  Direction {direction:2d}: {dataset.n_trials_per_direction[direction]} trials")

print(f"\nFree periods:")
print(f"  Free0: {dataset.free0_duration_sec:.1f}s ({dataset.free0_n_frames} frames)")
print(f"  Free1: {dataset.free1_duration_sec:.1f}s ({dataset.free1_n_frames} frames)")
print(f"  Intertrial: {dataset.intertrial_duration_sec:.1f}s ({dataset.intertrial_n_frames} frames)")

Animal: M103
Condition: normal
Available directions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

Trials per direction:
  Direction  0: 98 trials
  Direction  1: 68 trials
  Direction  2: 58 trials
  Direction  3: 102 trials
  Direction  4: 72 trials
  Direction  5: 58 trials
  Direction  6: 104 trials
  Direction  7: 84 trials
  Direction  8: 92 trials
  Direction  9: 98 trials
  Direction 10: 82 trials
  Direction 11: 78 trials

Free periods:
  Free0: 480.0s (48000 frames)
  Free1: 420.0s (42000 frames)
  Intertrial: 1518.0s (151800 frames)


## 3. Get Kinematics Data

In [10]:
# Get concatenated kinematics for direction 4, trial type 'trial'
direction = 4
left_foot = dataset.get_kinematics('left_foot', direction=direction, trial_type='trial')
print(f"Left foot kinematics (direction {direction}, trials):")
print(f"  Shape: {left_foot.shape}")
print(f"  X range: [{left_foot[:, 0].min():.1f}, {left_foot[:, 0].max():.1f}] cm")
print(f"  Y range: [{left_foot[:, 1].min():.1f}, {left_foot[:, 1].max():.1f}] cm")
print(f"  Z range: [{left_foot[:, 2].min():.1f}, {left_foot[:, 2].max():.1f}] cm")

Left foot kinematics (direction 4, trials):
  Shape: (21600, 3)
  X range: [-10.5, 23.9] cm
  Y range: [12.9, 20.4] cm
  Z range: [172.3, 213.0] cm


## 4. Get All Directions

In [11]:
# Get all directions concatenated
left_foot_all = dataset.get_kinematics('left_foot', direction=None, trial_type='trial')
print(f"Left foot (all directions):")
print(f"  Shape: {left_foot_all.shape}")
print(f"  Total frames: {left_foot_all.shape[0]}")
print(f"  Expected: {12 * dataset.n_trials_per_direction[0] * 600} frames (12 directions × ~50 trials × ~600 frames/trial)")

Left foot (all directions):
  Shape: (298200, 3)
  Total frames: 298200
  Expected: 705600 frames (12 directions × ~50 trials × ~600 frames/trial)


## 5. Align Kinematics to Perturbation Onset

In [12]:
# Align to perturbation onset
direction = 4
left_foot_concat = dataset.get_kinematics('left_foot', direction=direction, trial_type='trial')

aligned, perturb_idx = dataset.align_to_perturbation(
    left_foot_concat, 
    direction=direction,
    pre_ms=200, 
    post_ms=1500
)

print(f"Aligned left foot (direction {direction}):")
print(f"  Shape: {aligned.shape}")
print(f"  Expected: ({dataset.n_trials_per_direction[direction]}, 170, 3)")  # 200+1500 = 1700ms = 170 frames
print(f"  Perturbation frame index: {perturb_idx}")
print(f"  Expected: 20 (200ms / 10ms)")

# Compute statistics
mean, sem_vals = dataset.compute_statistics(aligned)
print(f"\nMean trajectory shape: {mean.shape}")
print(f"SEM shape: {sem_vals.shape}")
print(f"X coordinate at perturbation onset: {mean[perturb_idx, 0]:.2f} ± {sem_vals[perturb_idx, 0]:.2f} cm")

Aligned left foot (direction 4):
  Shape: (36, 170, 3)
  Expected: (72, 170, 3)
  Perturbation frame index: 20
  Expected: 20 (200ms / 10ms)

Mean trajectory shape: (170, 3)
SEM shape: (170, 3)
X coordinate at perturbation onset: 9.85 ± 0.86 cm


## 6. Test Spatial Centering

In [13]:
# Test spatial centering
print(f"Before centering:")
print(f"  X mean: {aligned[:, :, 0].mean():.4f} cm")
print(f"  Z mean: {aligned[:, :, 2].mean():.4f} cm")

aligned_centered = dataset.center_xz(aligned, axis=None)
print(f"\nAfter centering:")
print(f"  X mean: {aligned_centered[:, :, 0].mean():.4f} cm")
print(f"  Z mean: {aligned_centered[:, :, 2].mean():.4f} cm")
print(f"  (Should be close to 0)")

Before centering:
  X mean: 8.6717 cm
  Z mean: 196.4096 cm

After centering:
  X mean: -0.0000 cm
  Z mean: 0.0000 cm
  (Should be close to 0)


## 7. Test Continuous Data (Free Periods)

In [14]:
# Get free0 continuous data
free0_data = dataset.get_continuous_data('free0', 'left_foot')
print(f"Free0 left foot:")
print(f"  Shape: {free0_data.shape}")
print(f"  Duration: {free0_data.shape[0] * Params.BIN_SIZE:.2f}s")

# Extract a 5-second window
window = dataset.extract_window(free0_data, start_idx=0, duration_sec=5.0)
print(f"\n5-second window:")
print(f"  Shape: {window.shape}")
print(f"  Duration: {window.shape[0] * Params.BIN_SIZE:.2f}s")

Free0 left foot:
  Shape: (48000, 3)
  Duration: 480.00s

5-second window:
  Shape: (500, 3)
  Duration: 5.00s


## 8. Test Velocity Computation

In [15]:
# Compute velocity from aligned data
velocity = dataset.get_velocity(aligned)  # Shape: (n_trials, n_frames)
print(f"Velocity shape: {velocity.shape}")
print(f"Peak velocity across trials: {np.nanpercentile(velocity, 95):.1f} cm/s")
print(f"Mean velocity: {np.nanmean(velocity):.1f} cm/s")

# Velocity at perturbation onset
vel_at_onset = velocity[:, perturb_idx]
print(f"\nVelocity at perturbation onset:")
print(f"  Mean: {np.nanmean(vel_at_onset):.2f} cm/s")
print(f"  Std: {np.nanstd(vel_at_onset):.2f} cm/s")

Velocity shape: (36, 170)
Peak velocity across trials: 589.1 cm/s
Mean velocity: 312.1 cm/s

Velocity at perturbation onset:
  Mean: 346.73 cm/s
  Std: 131.90 cm/s


## 9. Test TimeSeriesPlotter Integration

In [3]:
from tools.behavior import TimeSeriesPlotter

# Create plotter for this session
plotter = TimeSeriesPlotter(dataset)
print(f"✓ TimeSeriesPlotter created for {dataset.session_name}")

# Test kinematics grid plot
direction = 4
fig, axes = plotter.plot_kinematics_grid(direction, trial_type='trial', pre_ms=200, post_ms=1500)
print(f"✓ Kinematics grid: {fig.get_size_inches()} inches, {axes.shape} subplots")
plt.close(fig)

# Test 2D trajectories plot
fig, axes = plotter.plot_trajectories_2d(direction, trial_type='trial')
print(f"✓ 2D trajectories: {fig.get_size_inches()} inches, {len(axes.flatten())} subplots")
plt.close(fig)

# Test statistics table
df_stats = plotter.get_statistics_table(direction, trial_type='trial')
print(f"✓ Statistics table: {len(df_stats)} rows, {len(df_stats.columns)} columns")
print(df_stats)

# Test statistics snapshot plot
fig, ax = plotter.plot_statistics_snapshot(direction, trial_type='trial')
print(f"✓ Statistics snapshot plot created")
plt.close(fig)

✓ TimeSeriesPlotter created for M103_2026_02_18_15_30
✓ Kinematics grid: [16. 12.] inches, (6, 3) subplots
✓ 2D trajectories: [15.  5.] inches, 6 subplots
✓ Statistics table: 6 rows, 7 columns
         Body Part X (mean) X (SEM) Y (mean) Y (SEM) Z (mean) Z (SEM)
0        Left Foot     9.85    0.85    16.51    0.21   196.40    1.00
1       Right Foot    -0.79    0.79    16.46    0.20   203.62    1.09
2       Hip Center     6.53    0.39    -3.38    0.14   200.23    0.34
3  Shoulder Center   -14.21    0.16    -3.95    0.03   174.28    0.15
4         Left Paw    -8.66    0.45    14.17    0.09   172.54    0.57
5        Right Paw   -16.43    0.51    13.66    0.13   177.61    0.68
✓ Statistics snapshot plot created
